# Brightway and headwater on ecoinvent

[Notebook 04](04-coming-from-brightway.ipynb) set Brightway and headwater side by side on a toy. This notebook does the same tasks on ecoinvent 3.12 cut-off, with EF 3.1 and the IPCC 2021 carbon indicators, and **times each one** on your machine. It is notebook 04's comparison at full size: read 04 first for what each line of code means.

**You need an ecoinvent licence.** Both tools read your local copy of ecoinvent's files and LCIA workbook. Nothing logs in to ecoinvent. This notebook is committed **without outputs**, since they hold ecoinvent values: clear them before you commit (`python notebooks/refresh_outputs.py --clear-licensed`).

**Versions.** bw2data 4.5.4, bw2calc 2.3, bw2io 0.9.14 and peewee 3.19.0, the `brightway` dependency group:

```bash
uv sync --group notebooks --group brightway
```

**Timings.** Each timing is printed with the machine's load average, the number of processes waiting for a CPU, averaged over the last minute. Above the number of CPUs the machine is busy and every time is longer. Times from one run are rough. The measured numbers of this repository, with their scope, are in [docs/appendix/benchmarks.md](../docs/appendix/benchmarks.md).

Most of the notebook's time is Brightway's import of ecoinvent, which runs once per Brightway folder.

## 1. Set up

Environment variables, set before you start Jupyter or VS Code:

| Variable | What it is |
|---|---|
| `ECOINVENT_DIR` | your ecoinvent 3.12 cut-off ecoSpold2 folder, with `datasets/` and `MasterData/` |
| `ECOINVENT_LCIA` | ecoinvent's `LCIA Implementation 3.12.xlsx` |
| `HEADWATER_HOME` | optional: headwater's release cache. Default `~/.headwater` |
| `NOTEBOOK_BRIGHTWAY_DIR` | optional: a folder for this notebook's Brightway data, kept between runs so the import runs once. Default: a new temporary folder, so Brightway imports ecoinvent on every run |

**Brightway never opens your own Brightway folder here.** Importing `bw2data` opens and migrates the folder `BRIGHTWAY2_DIR` names, so the cell sets it first, in the same cell, before the import. `NOTEBOOK_BRIGHTWAY_DIR` must be empty, or a folder this notebook made: it writes a marker file into it, and refuses any other folder.

The cell also defines `timed(task, tool)`, which measures the time of a block of code and keeps it, with the load average, for the summary at the end.

In [ ]:
import contextlib
import io
import logging
import os
import shutil
import subprocess
import sys
import tempfile
import time
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np

missing = [name for name in ["ECOINVENT_DIR", "ECOINVENT_LCIA"] if not os.environ.get(name)]
if missing:
    raise RuntimeError(f"Set {' and '.join(missing)}, then restart the kernel.")
ecoinvent_dir = Path(os.environ["ECOINVENT_DIR"]).expanduser()
lcia_workbook = Path(os.environ["ECOINVENT_LCIA"]).expanduser()
assert (ecoinvent_dir / "datasets").is_dir(), f"no datasets/ folder in {ecoinvent_dir}"
assert lcia_workbook.is_file(), f"{lcia_workbook} is not a file"

work = Path(tempfile.mkdtemp(prefix="brightway-and-headwater-"))
brightway_dir = Path(os.environ.get("NOTEBOOK_BRIGHTWAY_DIR") or work / "brightway").expanduser()
marker = brightway_dir / ".made-by-headwater-notebook-05"
brightway_dir.mkdir(parents=True, exist_ok=True)
if any(brightway_dir.iterdir()) and not marker.exists():
    raise RuntimeError(f"{brightway_dir} is not empty and was not made by this notebook.")
marker.touch()

os.environ["BRIGHTWAY2_DIR"] = str(brightway_dir)  # never your own Brightway folder
os.environ["BRIGHTWAY_NO_STRUCTLOG"] = "1"
os.environ["TQDM_DISABLE"] = "1"
logging.getLogger("brightway-stdout-feedback").addFilter(lambda record: False)
warnings.filterwarnings("ignore", module="bw2calc")

import bw2calc as bc  # noqa: E402
import bw2data as bd  # noqa: E402
import bw2io as bi  # noqa: E402

import headwater as hw  # noqa: E402
from headwater.release import releases_dir  # noqa: E402

bd.projects.set_current("ecoinvent-3.12")
assert Path(bd.projects.dir).resolve().is_relative_to(brightway_dir.resolve())

timings = defaultdict(dict)


@contextlib.contextmanager
def timed(task, tool):
    """Time the block, and keep the time and the load average for the summary."""
    start = time.perf_counter()
    yield
    seconds = time.perf_counter() - start
    load = os.getloadavg()[0]
    timings[task][tool] = (seconds, load)
    print(f"{tool}, {task}: {seconds:.2f} s (load average {load:.1f}, {os.cpu_count()} CPUs)")


HEADWATER = shutil.which("headwater", path=str(Path(sys.executable).parent))


def headwater(*args, quiet=False):
    """Run the headwater command and print what it prints: with `quiet`, only when it fails."""
    done = subprocess.run([HEADWATER, *map(str, args)], capture_output=True, text=True, cwd=work)
    if done.returncode or not quiet:
        print((done.stdout + done.stderr).rstrip())
    done.check_returncode()


print("Brightway folder:", brightway_dir)
print("headwater releases:", releases_dir())

## 2. Import ecoinvent and the methods

**Brightway.** These are the steps of bw2io's `import_ecoinvent_release`, without the download: that function fetches the files with your ecoinvent login, and this notebook reads the files you already have. First the elementary flows from `MasterData`, then the datasets, then bw2io's linking strategies, then the write into the project. A database already in the project is not imported again.

In [ ]:
version = "3.12"
biosphere_name = f"ecoinvent-{version}-biosphere"
database_name = f"ecoinvent-{version}-cutoff"

datasets_in_folder = len(list((ecoinvent_dir / "datasets").glob("*.spold")))
if database_name in bd.databases:
    print(f"{database_name} is already in this Brightway folder: not imported again")
else:
    with timed("import the database", "Brightway"):
        if not len(bi.migrations):
            bi.create_core_migrations()
        if biosphere_name not in bd.databases:
            biosphere = bi.importers.Ecospold2BiosphereImporter(
                name=biosphere_name,
                filepath=ecoinvent_dir / "MasterData" / "ElementaryExchanges.xml",
            )
            with contextlib.redirect_stdout(io.StringIO()):
                biosphere.apply_strategies()
            assert biosphere.all_linked
            biosphere.write_database(overwrite=False)
        bd.preferences["biosphere_database"] = biosphere_name
        with contextlib.redirect_stdout(io.StringIO()):
            datasets = bi.importers.SingleOutputEcospold2Importer(
                dirpath=ecoinvent_dir / "datasets",
                db_name=database_name,
                biosphere_database_name=biosphere_name,
                use_mp=True,
            )
            datasets.apply_strategies()
        assert datasets.all_linked
        datasets.write_database()
if len(bd.Database(database_name)) != datasets_in_folder:
    raise RuntimeError(
        f"{database_name} holds {len(bd.Database(database_name))} activities for "
        f"{datasets_in_folder} files: an earlier import stopped halfway. Empty {brightway_dir}."
    )
print(bd.databases)

Then the methods. bw2io reads every method of the LCIA workbook and finds each factor's flow by name and compartment. The function below follows bw2io 0.9.14's own code for that step (in `bw2io/ecoinvent.py`, which needs the `ecoinvent_interface` package and a login, so it is not imported here), reading only the three methods compared here. On 3.12 every factor finds its flow by exact name, so bw2io's fallback to a close name is left out: a factor that does not map stops the cell.

In [ ]:
from bw2io.extractors import ExcelExtractor  # noqa: E402


def header_dict(rows):
    """The rows of a sheet as dictionaries, keyed by the lower-case header (as bw2io does)."""
    return [
        {h.lower(): value for h, value in zip(rows[0], row, strict=False)}
        for row in rows[1:]
        if any(row)
    ]


def drop_unspecified(name, compartment, subcompartment):
    """Brightway's categories leave out a subcompartment `unspecified` (as bw2io does)."""
    if subcompartment.lower() == "unspecified":
        return (name, compartment)
    return (name, compartment, subcompartment)


compared = {
    "EF v3.1": None,  # every category
    "IPCC 2021 (incl. biogenic CO2)": {
        "climate change: biogenic removals (incl. CO2)",
        "climate change: biogenic emissions (incl. CO2)",
        "climate change: biogenic (incl. CO2)",
    },
    "IPCC 2021": {"climate change: direct land use change"},
}


def import_brightway_methods():
    data = dict(ExcelExtractor.extract(lcia_workbook))
    units = header_dict(data["Indicators"])
    unit_label = next(k for k in ("indicator unit", "unit", "unitname") if k in units[0])
    unit_of = {
        (f"ecoinvent-{version}", r["method"], r["category"], r["indicator"]): r[unit_label]
        for r in units
    }
    flow_of = {
        (node["name"],) + tuple(node["categories"]): node.id for node in bd.Database(biosphere_name)
    }

    factors = defaultdict(list)
    for row in header_dict(data["CFs"]):
        wanted = compared.get(row["method"], ())
        if row["cf"] is None or (wanted is not None and row["category"] not in wanted):
            continue
        key = (f"ecoinvent-{version}", row["method"], row["category"], row["indicator"])
        flow = drop_unspecified(row["name"], row["compartment"], row["subcompartment"])
        factors[key].append((flow_of[flow], float(row["cf"])))
    for key, cfs in factors.items():
        method = bd.Method(key)
        method.register(unit=unit_of.get(key, "Unknown"), filepath=str(lcia_workbook))
        method.write(cfs)


imported = {key[1] for key in bd.methods if key[0] == f"ecoinvent-{version}"}
if imported != set(compared):  # none yet, or an earlier import stopped halfway
    for key in [key for key in bd.methods if key[0] == f"ecoinvent-{version}"]:
        bd.Method(key).deregister()
    with timed("import the methods", "Brightway"):
        import_brightway_methods()
brightway_methods = sorted(key for key in bd.methods if key[0] == f"ecoinvent-{version}")
print(len(brightway_methods), "method tuples, for example", brightway_methods[0])

**headwater.** One command imports the database into a release in the cache, once per machine. If the cache already holds an `ecoinvent-3.12-cutoff` release, it is used and not timed. Then three commands write the three method folders, into the temporary folder.

In [ ]:
cached = sorted(
    releases_dir().glob("ecoinvent-3.12-cutoff@*"), key=lambda path: path.stat().st_mtime
)
if cached:
    print(f"Using {cached[-1].name}: already imported, not timed")
else:
    with timed("import the database", "headwater"):
        headwater("release", "import", ecoinvent_dir, "--version", version)
    cached = sorted(releases_dir().glob("ecoinvent-3.12-cutoff@*"))
release_ref = cached[-1].name

methods = work / "methods"
with timed("import the methods", "headwater"):
    headwater(
        "method",
        "import",
        lcia_workbook,
        "--method",
        "EF v3.1",
        "--flow-list",
        release_ref,
        "--out",
        methods / "EF-3.1",
        quiet=True,
    )
    headwater(
        "method",
        "import",
        lcia_workbook,
        "--method",
        "IPCC 2021 (incl. biogenic CO2)",
        "--category",
        "climate change: biogenic removals (incl. CO2)",
        "--category",
        "climate change: biogenic emissions (incl. CO2)",
        "--category",
        "climate change: biogenic (incl. CO2)",
        "--flow-list",
        release_ref,
        "--out",
        methods / "IPCC-2021-incl-biogenic-CO2",
        quiet=True,
    )
    headwater(
        "method",
        "import",
        lcia_workbook,
        "--method",
        "IPCC 2021",
        "--category",
        "climate change: direct land use change",
        "--flow-list",
        release_ref,
        "--out",
        methods / "IPCC-2021",
        quiet=True,
    )
print(release_ref, sorted(path.name for path in methods.iterdir()))

**Loading.** headwater's `hw.load` builds the matrices, factorizes them once and solves every product's score for the 29 indicators. Brightway does that work inside `lci()` and `lcia()`, timed in task 4.

In [ ]:
with timed("load and solve every score", "headwater"):
    bg = hw.load(
        release_ref,
        method=[methods / "EF-3.1", methods / "IPCC-2021-incl-biogenic-CO2", methods / "IPCC-2021"],
    )
print(bg.method.id, len(bg.method.indicators), "indicators")

**What differs.** The Brightway import writes ecoinvent into this project's SQLite database: another project, or another notebook with its own project, imports it again. The headwater release is written once into the cache and read by every database on the machine. Each method stays its own folder, with its own id, version and hash.

## 3. Find an activity

**Brightway.** `Database.search` asks its SQLite full-text index.

In [ ]:
database = bd.Database(database_name)
with timed("find a dataset", "Brightway"):
    found = database.search("market for textile woven cotton", limit=5)
for activity in found:
    print(activity)

**headwater.** `bg.find` checks names, then products, by exact rules. The first call builds its table in memory.

In [ ]:
with timed("find a dataset", "headwater"):
    matches = bg.find("market for textile, woven cotton", limit=5)
for match in matches:
    print(match.process_id, "|", match.name, "|", match.location, f"({match.why})")
textile = bg.find("market for textile, woven cotton", location="GLO", limit=1)[0]

Both tools name a dataset by the same two uuids, of its activity and of its product. headwater writes them in its id. bw2io keeps them as `activity` and `flow`, and makes the code from their md5 hash. These two functions go from one to the other:

In [ ]:
import hashlib  # noqa: E402


def headwater_id(node):
    return f"ecoinvent:{version}:cutoff:{node['activity']}_{node['flow']}"


def brightway_node(process_id):
    activity, product = process_id.rsplit(":", 1)[1].split("_")
    code = hashlib.md5((activity + product).encode()).hexdigest()
    return bd.get_node(database=database_name, code=code)


textile_node = brightway_node(textile.process_id)
print(textile_node, "=", textile.process_id)
assert headwater_id(textile_node) == textile.process_id

## 4. One footprint

1 kg of woven cotton textile, climate change of EF 3.1.

**Brightway.** The first `LCA` builds the matrices and factorizes them.

In [ ]:
climate = ("ecoinvent-3.12", "EF v3.1", "climate change", "global warming potential (GWP100)")
assert climate in bd.methods, climate
with timed("first footprint", "Brightway"):
    lca = bc.LCA({textile_node: 1}, method=climate)
    lca.lci(factorize=True)
    lca.lcia()
print(f"Brightway: {lca.score:.6g}")

**headwater.** The score was solved by `hw.load`. Reading it, with the tree of what it buys:

In [ ]:
with timed("first footprint", "headwater"):
    tree = bg.contributions(textile.process_id, indicator="climate-change")
print(f"headwater: {tree.score:.6g}")
difference = abs(lca.score - tree.score) / abs(tree.score)
print(f"relative difference: {difference:.1e}")
assert difference < 1e-6

The difference comes from Brightway's 32-bit storage of amounts and factors, as on the toy ([benchmarks](../docs/appendix/benchmarks.md#parity-with-brightway)).

## 5. Every footprint

**Brightway.** The way that reuses the work: the matrix is already factorized, so each further product is a new demand with `redo_lcia`, one solve each, for one method at a time. Scoring all 26,533 datasets this way takes a while, so the cell times a sample of 200, the same 200 on every run, and works out the rate. That last number is an estimate, not a measurement.

In [ ]:
rng = np.random.default_rng(12)
nodes = sorted(database, key=lambda node: node["code"])  # Brightway's order changes between runs
sample = [nodes[i] for i in rng.choice(len(nodes), size=200, replace=False)]
brightway_scores = {}
with timed("200 footprints, one indicator", "Brightway"):
    for node in sample:
        lca.redo_lcia({node.id: 1})
        brightway_scores[headwater_id(node)] = lca.score
seconds = timings["200 footprints, one indicator"]["Brightway"][0]
per_product = seconds / 200
print(f"{per_product * 1000:.1f} ms per product")
print(f"estimate for all {len(nodes):,}: {per_product * len(nodes) / 60:.0f} min per indicator")

**headwater.** Every product's score for every indicator is already solved. `hw.score_many` reads all 26,533 of them, for 29 indicators, in one call.

In [ ]:
ids = bg.release.table("processes")["id"].to_pylist()
with timed("every footprint, every indicator", "headwater"):
    every = hw.score_many(bg, {"sku": ids, "product_id": ids, "amount": [1.0] * len(ids)})
print(every.scores.shape, "(products, indicators)")

Compare the 200. Brightway's demand of +1 for a waste treatment is minus one kg treated, so its score has the opposite sign of headwater's per kg treated ([notebook 04](04-coming-from-brightway.ipynb)). The cell turns the sign of those, the processes whose product is a waste, and compares the others as they are.

In [ ]:
waste_query = (
    "select p.id from processes p join flows f on f.id = p.reference_flow_id where f.kind = 'waste'"
)
treatments = {pid for (pid,) in bg.release.connect().sql(waste_query).fetchall()}
column = every.indicators.index("climate-change")
row_of = {sku: i for i, sku in enumerate(every.skus)}
ours = np.array([every.scores[row_of[pid], column] for pid in brightway_scores])
theirs = np.array(
    [-score if pid in treatments else score for pid, score in brightway_scores.items()]
)
nonzero = ours != 0
relative = np.abs(theirs[nonzero] - ours[nonzero]) / np.abs(ours[nonzero])
print(f"{len(ours)} products, {sum(pid in treatments for pid in brightway_scores)} treatments")
print(f"{nonzero.sum()} non-zero scores, largest relative difference {relative.max():.1e}")

**What differs.** Brightway solves the system once per product, reusing the factorization: the time grows with the number of products, and again with the number of methods. headwater solves the transposed system once per indicator, which gives every product's score at once, inside `hw.load`. `MultiLCA` batches demands and methods in Brightway, but each demand is still one solve.

## 6. Contribution analysis

Which processes emit the textile's climate change score?

**Brightway.** Sum the characterised inventory by activity, as bw2analyzer does, and keep the 10 largest.

In [ ]:
with timed("top 10 emitters", "Brightway"):
    lca.redo_lcia({textile_node.id: 1})
    by_activity = np.asarray(lca.characterized_inventory.sum(axis=0)).ravel()
    activity_of = {column: node_id for node_id, column in lca.dicts.activity.items()}
    top = np.argsort(-np.abs(by_activity))[:10]
    brightway_top = [headwater_id(bd.get_node(id=activity_of[c])) for c in top]
for c in top:
    node = bd.get_node(id=activity_of[c])
    print(f"{by_activity[c]:10.4g}  {node['name']} ({node['location']})")

**headwater.** `bg.where` gives the same split, with how much each process runs and the sum by location.

In [ ]:
with timed("top 10 emitters", "headwater"):
    where = bg.where(textile.process_id, indicator="climate-change", top=10)
print(where)
headwater_top = [emitter.process_id for emitter in where.processes]
print("same 10 processes:", set(headwater_top) == set(brightway_top))
print("same order:       ", headwater_top == brightway_top)

On 10 products and 2 indicators this was the same top 10, in the same order, in 20 of 20 checks ([benchmarks](../docs/appendix/benchmarks.md#top-contributors-against-brightway)). Two processes with nearly equal parts can swap places, because of Brightway's 32-bit storage.

## 7. Change one input and recompute

A tote bag buys 0.15 kg of woven cotton textile and 0.1 kWh of French medium-voltage electricity. Make it 0.18 kg of textile.

**Brightway.** The tote is an activity in a foreground database. After the edit, a new `LCA` builds and factorizes the matrices again.

In [ ]:
electricity = bg.find("market for electricity, medium voltage", location="FR", limit=1)[0]
electricity_node = brightway_node(electricity.process_id)
bd.Database("tote").write(
    {
        ("tote", "tote"): {
            "name": "tote bag",
            "reference product": "tote bag",
            "unit": "unit",
            "location": "FR",
            "exchanges": [
                {"input": ("tote", "tote"), "amount": 1, "type": "production"},
                {"input": textile_node.key, "amount": 0.15, "type": "technosphere"},
                {"input": electricity_node.key, "amount": 0.1, "type": "technosphere"},
            ],
        }
    }
)
tote = bd.get_node(database="tote", code="tote")
edge = next(e for e in tote.technosphere() if e.input.id == textile_node.id)
edge["amount"] = 0.18
edge.save()
with timed("change one input and recompute", "Brightway"):
    changed = bc.LCA({tote: 1}, method=climate)
    changed.lci()
    changed.lcia()
print(f"Brightway: {changed.score:.6g}")

**headwater.** The tote is a database folder. The edit is one line of its CSV file, and `calc` uses the stored scores, with no new solve of the background.

In [ ]:
flow_of = dict(bg.release.connect().sql("select id, reference_flow_id from processes").fetchall())
tote_folder = work / "databases/tote"
(tote_folder / "processes").mkdir(parents=True)
(tote_folder / "database.yaml").write_text(f"""\
name: tote
background: {release_ref}
methods: [EF-3.1, IPCC-2021-incl-biogenic-CO2, IPCC-2021]
""")
(tote_folder / "flows.csv").write_text("id,name,kind,unit\ntote:tote-bag,tote bag,product,unit\n")
(tote_folder / "processes/tote.yaml").write_text(
    "name: tote bag\nlocation: FR\nkind: ordinary_transforming_activity\n"
)
(tote_folder / "processes/tote.csv").write_text(f"""\
line,role,flow,amount,unit,supplier
e01,reference,tote:tote-bag,1,unit,
e02,input,{flow_of[textile.process_id]},0.15,kg,{textile.process_id}
e03,input,{flow_of[electricity.process_id]},0.1,kWh,{electricity.process_id}
""")
lines = tote_folder / "processes/tote.csv"
lines.write_text(lines.read_text().replace(",0.15,", ",0.18,"))
with timed("change one input and recompute", "headwater"):
    result = hw.open(tote_folder).calc(bg, "tote:tote", indicator="climate-change")
print(f"headwater: {result.score:.6g}")
print(f"relative difference: {abs(changed.score - result.score) / abs(result.score):.1e}")

**What differs.** Brightway changes the project in place and factorizes ecoinvent again for the new system. headwater changes a text file, which git records, and scores the tote from the stored scores. `result.sensitivity()` would have predicted the change before it was made.

## 8. Import an Excel inventory

The same tote, written in Brightway's Excel format, buying from the database `ecoinvent-3.12-cutoff` by name, product and location.

In [ ]:
import openpyxl  # noqa: E402

rows = [
    ["Database", "tote-excel"],
    [],
    ["Activity", "tote bag"],
    ["production amount", 1],
    ["reference product", "tote bag"],
    ["location", "FR"],
    ["unit", "unit"],
    ["Exchanges"],
    ["name", "amount", "unit", "location", "type", "database", "reference product"],
    ["tote bag", 1, "unit", "FR", "production", "tote-excel", "tote bag"],
    [
        "market for textile, woven cotton",
        0.15,
        "kilogram",
        "GLO",
        "technosphere",
        database_name,
        "textile, woven cotton",
    ],
    [
        "market for electricity, medium voltage",
        0.1,
        "kilowatt hour",
        "FR",
        "technosphere",
        database_name,
        "electricity, medium voltage",
    ],
]
workbook = openpyxl.Workbook()
for row in rows:
    workbook.active.append(row)
xlsx = work / "tote.xlsx"
workbook.save(xlsx)

**Brightway.**

In [ ]:
if "tote-excel" in bd.databases:  # from an earlier run in a kept Brightway folder
    del bd.databases["tote-excel"]
with timed("import an Excel inventory", "Brightway"):
    with contextlib.redirect_stdout(io.StringIO()):
        importer = bi.ExcelImporter(xlsx)
        importer.apply_strategies()
        fields = ("name", "reference product", "unit", "location")
        importer.match_database(database_name, fields=fields)
        importer.match_database(fields=fields)
    importer.statistics()
    importer.write_database()
excel_lca = bc.LCA({bd.get_node(database="tote-excel", name="tote bag"): 1}, method=climate)
excel_lca.lci()
excel_lca.lcia()
print(f"Brightway: {excel_lca.score:.6g}")

**headwater.**

In [ ]:
with timed("import an Excel inventory", "headwater"):
    headwater(
        "database",
        "import-excel",
        xlsx,
        "--release",
        release_ref,
        "--out",
        work / "databases/tote-excel",
        "--name",
        "tote-excel",
        "--method",
        methods / "EF-3.1",
    )

`import-excel` takes one method, and writes it in `database.yaml`. A database that lists methods must list every method of the background it is calculated on, in the same order, so add the two IPCC methods there. The import names the tote's process after the activity's name and location: `tote-excel:tote-bag-fr`.

In [ ]:
excel_yaml = work / "databases/tote-excel/database.yaml"
excel_yaml.write_text(
    excel_yaml.read_text().replace(
        "methods: [EF-3.1]", "methods: [EF-3.1, IPCC-2021-incl-biogenic-CO2, IPCC-2021]"
    )
)
assert "methods: [EF-3.1, " in excel_yaml.read_text()
excel_result = hw.open(work / "databases/tote-excel").calc(
    bg, "tote-excel:tote-bag-fr", indicator="climate-change"
)
print(f"headwater: {excel_result.score:.6g}")
difference = abs(excel_lca.score - excel_result.score) / abs(excel_result.score)
print(f"relative difference: {difference:.1e}")

## Summary of the timings

One run, on this machine, with the load average when each step ended. A blank is a step that did not run: an import that was already done, or a step one tool does not have.

In [ ]:
print(f"{'task':36} {'Brightway':>22} {'headwater':>22}")
for task, tools in timings.items():
    cells = []
    for tool in ["Brightway", "headwater"]:
        if tool in tools:
            seconds, load = tools[tool]
            cells.append(f"{seconds:9.2f} s (load {load:4.1f})")
        else:
            cells.append("")
    print(f"{task:36} {cells[0]:>22} {cells[1]:>22}")
print(f"{os.cpu_count()} CPUs")

## What Brightway has that headwater does not, yet

As in [notebook 04](04-coming-from-brightway.ipynb): more import formats (bw2io also reads EXIOBASE), regionalised and temporal LCA, a graphical interface (the Activity Browser), and computed allocation for your own multi-output processes.

Before you commit this notebook, clear its outputs: `uv run --group notebooks python notebooks/refresh_outputs.py --clear-licensed`.